# Kashmiri word OCR: characters vs. PAW units

Trains a CNN + BiLSTM + CTC word recognizer for printed Kashmiri on:

* **Kashmiri OCR** by nawabhussaen (69,434 labelled word images, mostly Naskh-style fonts). Licence on Kaggle: *unknown*, so use for research only and ask the author before redistributing anything derived from it.
* **Synthetic Nastaliq + Naskh words** drawn here from the 56,000-word Kashmiri dictionary export (`public/lexicon/ks-lexicon.json` in github.com/Rizvonium/kashmiri_ocr), with camera-like damage.

It trains twice, once with **characters** as output units and once with **PAW units** (every piece of a word between non-joining letters, e.g. کٲشُر = کٲ + شُر, seen at least 25 times; rarer pieces fall back to characters). Test words are never seen in training in any font.

**Before running:** Settings → Accelerator **GPU**, Settings → **Internet on**, and *Add Input* → search **kashmiri-ocr** (by nawabhussaen).

In [ ]:
!git clone --depth 1 https://github.com/Rizvonium/kashmiri_ocr.git /kaggle/working/kashmiri_ocr
%cd /kaggle/working/kashmiri_ocr
!pip install -q fonttools onnx
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the accelerator to GPU")

In [ ]:
# Find the attached Kaggle dataset (its mount path differs between Kaggle versions)
import glob, os
hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/labels", recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(p), "word_images"))]
assert hits, "Attach the 'kashmiri-ocr' dataset by nawabhussaen (Add Input)"
KAGGLE_DATA = hits[0]; print("dataset:", KAGGLE_DATA, "-", len(os.listdir(os.path.join(KAGGLE_DATA, "labels"))), "labels")

In [ ]:
# Open-licence Kashmiri-capable fonts (SIL OFL) from the Google Fonts repository
FONTS = {
  "NotoNastaliqUrdu.ttf": "ofl/notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf",
  "Gulzar-Regular.ttf": "ofl/gulzar/Gulzar-Regular.ttf",
  "NotoNaskhArabic.ttf": "ofl/notonaskharabic/NotoNaskhArabic%5Bwght%5D.ttf",
  "ScheherazadeNew-Regular.ttf": "ofl/scheherazadenew/ScheherazadeNew-Regular.ttf",
}
os.makedirs("fonts", exist_ok=True)
for name, path in FONTS.items():
    !curl -sSL -o fonts/{name} https://cdn.jsdelivr.net/gh/google/fonts@main/{path}
!ls -la fonts

In [ ]:
# Arabic letters only join if Pillow has libraqm; install the system libraries if it does not
from PIL import features
if not features.check("raqm"):
    !apt-get -qq install -y libraqm0 libfribidi0 > /dev/null
    print("installed libraqm; RESTART the session (Run > Restart) and run from the top if the next line says False")
import importlib, PIL.features; importlib.reload(PIL.features); print("raqm available:", PIL.features.check("raqm"))

In [ ]:
# 80,000 synthetic word images, 70% in Nastaliq fonts (about 5 minutes)
!python training/synth_words.py --fonts fonts --out /kaggle/working/synth --n 80000 --nastaliq-share 0.7

In [ ]:
# Run 1: character units
!python training/train_word_ocr.py --data {KAGGLE_DATA} /kaggle/working/synth --units chars --epochs 25 --batch 128 --workers 4 --out /kaggle/working/runs/chars

In [ ]:
# Run 2: PAW units (the expanded space), same data and split
!python training/train_word_ocr.py --data {KAGGLE_DATA} /kaggle/working/synth --units paws --epochs 25 --batch 128 --workers 4 --out /kaggle/working/runs/paws

In [ ]:
# Compare on unseen test words
import json, pandas as pd
rows = []
for run in ("chars", "paws"):
    r = json.load(open(f"/kaggle/working/runs/{run}/report.json"))
    for group, v in r["test"].items():
        rows.append({"units": run, "outputs": r["n_outputs"], "group": group, "n": v["n"], "CER": round(v["CER"], 4),
                     "WER": round(v["WER"], 4), "WER + dictionary": None if v["WER_with_dictionary"] is None else round(v["WER_with_dictionary"], 4)})
df = pd.DataFrame(rows).pivot_table(index="group", columns="units", values=["CER", "WER", "WER + dictionary"])
df

**Outputs** (in `/kaggle/working/runs/<units>/`): `best.pt` (PyTorch), `kashmiri_word_ocr.onnx` + `kashmiri_word_ocr.json` (for the web app via onnxruntime-web), and `report.json`. Download them from the *Output* tab, or *Save Version* to keep them.